In [3]:
import pandas as pd
import requests
import comtradeapicall
import os
from dotenv import load_dotenv
import itertools 

In [16]:
load_dotenv()
api_key = os.environ.get('primary_key', None)

In [42]:
date_range = pd.date_range(start='2018-01', end='2025-12', freq='MS') 

# Convert the datetime objects to the YYYYMM string format
date_series = date_range.strftime('%Y%m').tolist()
date_series_int = [int(date_str) for date_str in date_series]

date_series_batched = list(itertools.batched(date_series_int,8))

In [43]:
dataframes_list = []

for date_period in date_series_batched:
    # This is how the API wants the date to look like
    api_date_string = ",".join(str(date_int) for date_int in date_period)
    trade_data = comtradeapicall.getFinalData(
        api_key,
        typeCode="C",
        freqCode="M",
        clCode="HS",
        period=api_date_string,
        # period="201801,201802,201803,201804,201805,201806,201807,201808,201809,201810,201811,201812",
        reporterCode="97",
        cmdCode="8414,8415,8418,8450,8508,8509,8516,9401,9403,9405,9501,9502,9503,3923,3924,3926",
        flowCode="M",
        partnerCode=156,
        partner2Code=0,
        customsCode=None,
        motCode=None,
        maxRecords=2500,
        format_output="JSON",
        aggregateBy=None,
        breakdownMode="classic",
        countOnly=None,
        includeDesc=True,
    )
    dataframes_list.append(trade_data)

In [44]:
df_concat = pd.concat(dataframes_list)

In [45]:
df_concat.tail()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
65,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,9.794778e+07,False,0.0,False,7.316084e+08,None,7.316084e+08,2,False,True
66,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,1.202756e+08,True,0.0,False,5.002984e+08,None,5.002984e+08,6,False,True
67,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,2.192306e+08,False,0.0,False,5.343282e+08,None,5.343282e+08,2,False,True
68,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,7.686484e+07,True,0.0,False,8.195563e+08,None,8.195563e+08,6,False,True
69,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,8.777624e+07,True,0.0,False,8.313436e+08,None,8.313436e+08,6,False,True


In [46]:
df_concat.to_csv('data/trade_data_2018_2025.csv', index=False)